# Session 4 — Statistical models, academic outputs & library map  
**Duration:** 1.5 hours

### Learning goals
1. Build paper-style **Mean (SD)** tables.
2. Run a transparent **t-test** with Cohen’s *d* and an APA-like sentence.
3. Fit a simple **regression / group model** with statsmodels or pingouin.
4. Leave with a **library roadmap** for continued learning.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SESSION_DIR = Path.cwd()
WORKSHOP_DIR = SESSION_DIR.parent if SESSION_DIR.name == "sessions" else SESSION_DIR / "workshop"
sys.path.insert(0, str(WORKSHOP_DIR))

from analysis.paths import data_path
from analysis.stats_report import mean_sd_table, apa_ttest

# Optional richer APIs
import pingouin as pg
import statsmodels.formula.api as smf

## 1. Prepare an analysis-ready metrics table (15 min)

In [ ]:
gsr = pd.read_csv(data_path("tobii_gsr_demo", "Tobii_Pro_Lab_GSR_Demo_Project_Metrics.tsv"), sep="\t")
# Example: dwell-like metric on Snake AOI if present
col = "Total_duration_of_whole_fixations.Snake"
if col not in gsr.columns:
    # fallback: first matching duration column
    cand = [c for c in gsr.columns if c.startswith("Total_duration_of_whole_fixations.")]
    col = cand[0]
    print("Using", col)

df = gsr[["Recording", "Participant", "TOI", "Media", col, "Average_GSR", "Average_whole-fixation_pupil_diameter"]].copy()
df = df.rename(columns={col: "dwell_snake"})
for c in ["dwell_snake", "Average_GSR", "Average_whole-fixation_pupil_diameter"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["dwell_snake"])
df.head()

## 2. Descriptive table — Mean (SD) (15 min)

In [ ]:
desc = mean_sd_table(df, "dwell_snake", by="TOI")
desc

## 3. Inferential test for training (20 min)

In [ ]:
# Split TOIs into two groups if possible; otherwise demonstrate on median split of GSR
toi_counts = df["TOI"].value_counts()
print(toi_counts.head())
if toi_counts.shape[0] >= 2:
    a_name, b_name = toi_counts.index[:2]
    a = df.loc[df["TOI"] == a_name, "dwell_snake"]
    b = df.loc[df["TOI"] == b_name, "dwell_snake"]
else:
    a_name, b_name = "low_GSR", "high_GSR"
    med = df["Average_GSR"].median()
    a = df.loc[df["Average_GSR"] <= med, "dwell_snake"]
    b = df.loc[df["Average_GSR"] > med, "dwell_snake"]

result = apa_ttest(a, b, label_a=str(a_name), label_b=str(b_name))
result

In [ ]:
print(result["apa"])
# pingouin version (nice for teaching)
pg.ttest(a, b, correction=True)

## 4. Regression model with statsmodels (20 min)

In [ ]:
model_df = df.dropna(subset=["Average_GSR", "Average_whole-fixation_pupil_diameter"]).copy()
model_df = model_df.rename(columns={"Average_whole-fixation_pupil_diameter": "pupil"})
# Predict dwell from GSR + pupil (illustrative — discuss causality!)
fit = smf.ols("dwell_snake ~ Average_GSR + pupil", data=model_df).fit()
print(fit.summary())

### How to read this in an academic workshop
- **Coefficient**: expected change in dwell for +1 unit predictor (holding others constant).
- **Std. Err. / t / P>|t|**: uncertainty and null-hypothesis test.
- **R²**: variance explained — not proof of theory.
- Always report **N**, design (TOI/AOI definitions), and preprocessing (I-VT thresholds, blink handling).


## 5. Library roadmap (15 min)

### Core (this workshop)
`numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`, `statsmodels`, `pingouin`, `Pillow`, `openpyxl`

### Eye-tracking ecosystem (know the names)
| Tool | Typical use |
|------|-------------|
| Vendor exports (Tobii Pro Lab, Pupil Player) | What we used today |
| **PyGaze** / analysis add-ons | Experiment + basic analysis |
| **eyelinkio** / SR Research tools | EyeLink EDF access |
| **neurokit2** | GSR/EDA, ECG feature pipelines |
| **MNE-Python** | Mostly EEG/MEG, but great for epochs mindset |
| Custom I-VT / I-DT | Transparent methods sections |

### Reproducibility habits
- Keep a `requirements.txt`
- Save analysis-ready CSVs
- Never overwrite raw exports
- Document AOI polygons and I-VT thresholds in the paper/supplement


In [ ]:
# Capstone sketch: participant-level table for "publication demo"
cap = (
    df.groupby("Participant", as_index=False)
    .agg(
        n_rows=("dwell_snake", "size"),
        dwell_snake_mean=("dwell_snake", "mean"),
        gsr_mean=("Average_GSR", "mean"),
        pupil_mean=("Average_whole-fixation_pupil_diameter", "mean"),
    )
)
cap.head()

## Capstone (remaining time)

In pairs, produce **one figure + one APA-like sentence** using either:
- Food decision AOI dwell, or
- GSR demo dwell / GSR / pupil

### Exit ticket
Submit: figure filename idea + the statistical sentence + which library produced it.
